In [7]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="mxbai-embed-large",
)

In [2]:
from langchain_community.document_loaders import JSONLoader
from langchain_community.document_loaders import DirectoryLoader

json_loader = DirectoryLoader(
    path="./data/data",
    glob="**/*.json",
    loader_cls=JSONLoader,
    loader_kwargs={"jq_schema" : "..", "text_content" : False}
)
json_documents = json_loader.load()

In [ ]:
# Use this for PDFs:
"""
from document_processor import DocumentProcessor
from pdf_to_rag_example import run_complete_pipeline

# Process PDFs with automatic chunking
qdrant = run_complete_pipeline(
    data_path="./data/pdfs/",           # Your PDF directory
    collection_name="PDF_Knowledge",    # Collection name
    document_type="pdf"                 # Handles PyPDF + chunking automatically
)
"""

In [10]:
len(json_documents)

5828

In [9]:
# QUICK FIX: Connect to existing Qdrant collection (no 27-minute wait!)
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient

try:
    print("🔍 Connecting to existing DnD_Documents collection...")
    
    # Connect to your existing collection
    url = "127.0.0.1:6334"
    client = QdrantClient(url=url, prefer_grpc=True)
    
    # Check if collection exists
    if client.collection_exists("DnD_Documents"):
        print("✅ Found existing collection!")
        
        # Connect without recreating
        qdrant = QdrantVectorStore(
            client=client,
            collection_name="DnD_Documents",
            embedding=embeddings
        )
        
        # Get collection info
        count = client.count("DnD_Documents").count
        print(f"📊 Collection contains {count} documents")
        print("🎉 Ready for RAG operations!")
        
    else:
        print("❌ Collection 'DnD_Documents' not found")
        print("🔨 Creating new collection...")
        
        # Create new if doesn't exist
        qdrant = QdrantVectorStore.from_documents(
            json_documents,
            embeddings,
            url=url,
            prefer_grpc=True,
            collection_name="DnD_Documents",
        )
        print("✅ New collection created!")
        
except Exception as e:
    print(f"❌ Connection error: {e}")
    print("💡 Make sure Qdrant is running: docker ps | grep qdrant")


🔍 Connecting to existing DnD_Documents collection...
✅ Found existing collection!
📊 Collection contains 8388 documents
🎉 Ready for RAG operations!


In [ ]:
# Test that qdrant variable is now defined and working
try:
    print("🧪 Testing qdrant variable...")
    print(f"✅ qdrant is defined: {type(qdrant)}")
    print(f"Collection has {qdrant._client.count('DnD_Documents').count} documents")
    
    
    # Test similarity search
    test_results = qdrant.similarity_search("dragon", k=3)
    print(f"🔍 Found {len(test_results)} similar documents")
    
    if test_results:
        print(f"📋 First result preview: {test_results[0].page_content[:100]}...")
    
    print("🎉 SUCCESS: qdrant variable is working!")
    
except NameError:
    print("❌ qdrant is still not defined - run the cell above first")
except Exception as e:
    print(f"⚠️ qdrant defined but error occurred: {e}")


> NOTE: This cell will take a while (~15min) as there are a lot of documents - please ensure you'd selected your own data before running this cell.

In [ ]:
"""
from langchain_qdrant import QdrantVectorStore

url = "127.0.0.1:6334"
qdrant = QdrantVectorStore.from_documents(
    json_documents,
    embeddings,
    url=url,
    prefer_grpc=True,
    collection_name="DnD_Documents",
)
"""